# 02 · Productos — qué añadir y en qué orden

Notebook vivo de exploración. Cada sección: **código → conclusión**. Las conclusiones van en celdas markdown para poder consultarlas sin re-ejecutar.

**Pregunta.** Sobre las 1.286 empresas del dataset, de los 5 SKUs de la pestaña Productos de la SPA, ¿cuáles tienen sentido, con qué evidencia, y en qué orden de prioridad?

**Grupo primero.** Empresas del mismo `group_id` con excedente y agujero no compran yield ni crédito: mueven capital. El neteo está en la sección 3b y en `analysis/productos.py`.

**Qué no es este notebook.** No cambia `PRODUCT_DEFS` ni la SPA. No reescribe `context/monetizacion.md`. No usa el proxy `n_tx > 50` como elegibilidad FX. No escala 1.286 empresas sintéticas a «500 equipos Embat». El módulo SaaS a 350 €/mes queda fuera: ya está en el pitch y no es un SKU de Productos.

**Requisitos.** `duckdb`, `polars`, `plotly`. Capa de mapeo + caja reconstruida (`analysis/cash.duckdb` si existe; si no, se reconstruye en memoria, ~30 s).

**Artefacto HTML.** `python analysis/productos.py` escribe `analysis/productos.html` (autocontenido, se abre sin servidor).



In [ ]:
import sys
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "analysis"))

from monetizacion import (  # noqa: E402
    ADOPTION,
    ARTIFACT_EUR,
    DEPOSIT_BPS,
    FX_BPS,
    WINDOW_END,
    WINDOW_START,
    connect,
)

pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(140)
pl.Config.set_fmt_str_lengths(48)
pl.Config.set_fmt_float("mixed")

COLORS = {
    "navy": "#102a43",
    "blue": "#1463ff",
    "cyan": "#27b3c2",
    "green": "#22a06b",
    "amber": "#f59e0b",
    "red": "#e5484d",
    "slate": "#64748b",
}

YIELD_BPS = DEPOSIT_BPS          # 100 pb
RESERVE_BPS = 0.005              # 50 pb (SPA)
FACTORING_BPS = 0.015            # 1,5 % descuento (SPA)
INSURANCE_PREM = 0.005           # 0,5 % del inflow × late_share
INSURANCE_REFERRAL = 0.20        # 20 % de la prima
CFO_GROSS = 0.025                # colocación 2,5 %
AS_OF = "2026-09-01"
CAP = ARTIFACT_EUR               # tope por empresa = umbral de artefacto 100 M€

con = connect()


def q(sql: str) -> pl.DataFrame:
    return con.execute(sql).pl()


def money(x, decimals=0):
    if x is None:
        return "—"
    return f"{x:,.{decimals}f} €".replace(",", " ")


def meur(x, decimals=2):
    if x is None:
        return "—"
    return f"{x:.{decimals}f} M€"


try:
    from IPython.display import display
except ImportError:
    display = print

print("conectado", ROOT.name, "| artefacto", ARTIFACT_EUR, "| adopción", ADOPTION)


## 1. Premisas

Catálogo (ids de `research/src/service.py` `PRODUCT_DEFS`):

| id | SKU | Take Embat | Valor CFO | Clase de cifra |
|---|---|---|---|---|
| `yield` | Colocación de excedente | 100 pb sobre el ocioso | 2,5 % bruto; ~1,5 % neto | **Medida** la base; hipótesis el margen y la adopción |
| `fx` | Cobertura / ejecución de divisa | 15 pb sobre notional anual EUR | el mismo spread evitado | **Medida** la base; hipótesis el margen y la adopción |
| `reserve` | Financiación preventiva | 50 pb sobre la línea | runway comprado | Elegibilidad tesorera medida; **notional de línea no defendible sin tope** |
| `factoring` | Anticipo de AR vencido | 1,5 % del vencido | caja hoy ≈ 98,5 % | AR **medido**; take y que se factorice, hipótesis. Es un one-shot, no una renta |
| `credit` | Seguro de impago | 20 % de una prima 0,5 % × inflow × late_share | prima dinámica | Elegibilidad medida; prima **hipótesis apilada** |

Reglas de lectura:

- **Medido** = sale de mapping + caja reconstruida + facturas, en EUR, sin centinelas ≥ 100 M€, sin series `has_drift`.
- **Si asumimos** = adopción 25/35/45 % y los bps de la SPA / monetización. Nunca se presentan como venta observada.
- El crédito **no lidera el pitch**, aunque el n sea alto. Esa es la regla de `context/monetizacion.md`.
- Yield y FX no se canibalizan (bases distintas). Yield y reserve sí: quien tiene excedente coloca, no origina una línea. Factoring y seguro comparten la cola de AR.



## 2. Universo y cobertura



In [ ]:
universe = q("""
SELECT
  (SELECT count(*) FROM companies) AS n_companies,
  (SELECT count(*) FROM cash.cash_summary) AS n_panel,
  (SELECT count(*) FROM cash.cash_summary WHERE NOT has_drift) AS n_reliable,
  (SELECT count(*) FROM cash.cash_summary WHERE has_drift) AS n_drift,
  (SELECT count(*) FROM companies WHERE has_invoices) AS n_erp,
  (SELECT count(*) FROM companies WHERE NOT has_invoices) AS n_no_erp
""")
display(universe)

print("\ndiagnóstico de caja (último mes observado)")
display(q("""
SELECT diagnosis, count(*) AS n,
       count(*) FILTER (WHERE NOT has_drift) AS n_fiables
FROM cash.cash_summary
GROUP BY 1
ORDER BY n DESC
"""))

print("\ndeuda ya contratada (foto final, sin dimensión temporal)")
display(q("""
SELECT type, count(*) AS n_productos, count(DISTINCT company_id) AS n_empresas
FROM debt_products
GROUP BY 1
ORDER BY n_empresas DESC
"""))

print("\ncuentas saving")
display(q("""
SELECT count(*) AS n_productos, count(DISTINCT company_id) AS n_empresas
FROM banking_products WHERE type = 'saving'
"""))

print("\ncobertura en meses de gasto (solo fiables)")
display(q("""
SELECT CASE
         WHEN cash_eur < 0 THEN '1. caja negativa'
         WHEN coverage < 0.25 THEN '2. colchón crítico (<0,25 m)'
         WHEN coverage < 2 THEN '3. fino (0,25–2 m)'
         WHEN coverage < 6 THEN '4. 2–6 meses'
         ELSE '5. ≥ 6 meses'
       END AS tramo,
       count(*) AS n
FROM cash.cash_summary
WHERE NOT has_drift
GROUP BY 1
ORDER BY 1
"""))


### Conclusión — universo

- **1.286** empresas; **1.249** con serie de caja reconstruida; **1.199** fiables (`NOT has_drift`). Las 50 con deriva y las 37 sin panel **no entran** en colas de yield/reserve.
- **785** con ERP/facturas, **501** sin ellas. Factoring y seguro son productos de observabilidad: no se pueden recomendar sin facturas, y eso no es un juicio de salud.
- Deuda ya existente (foto final): préstamo 239 empresas, póliza (`lineofcredit`) **206**, confirming 70, factoring **19**, saving **9**. El barrido de excedentes no tiene riel: casi nadie tiene cuenta de ahorro.
- Diagnóstico fiable: 438 estables, 331 colchón crítico, 158 mejora, 135 deterioro estructural, 96 bache recuperado, **41 caja negativa**. La cola de déficit es pequeña; la de colchón fino es la mayoría.



## 3. Panel de necesidad `company_id × SKU`

Una fila por empresa. FX e invoices se convierten a EUR con `cash.fx` (misma pila que el excedente). Las consultas pesadas se materializan una vez.



In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fx_co AS
WITH prod AS (
    SELECT product_id, currency FROM banking_products
    UNION ALL
    SELECT product_id, currency FROM debt_products
)
SELECT t.company_id,
       coalesce(sum(abs(t.amount) / coalesce(f.units_per_eur, 1)) FILTER (
           WHERE p.currency IS NOT NULL AND p.currency <> c.currency
             AND abs(t.amount / coalesce(f.units_per_eur, 1)) < {ARTIFACT_EUR}
             AND f.currency IS NOT NULL
       ), 0) / 2 AS fx_annual_eur,
       count(*) FILTER (
           WHERE p.currency IS NOT NULL AND p.currency <> c.currency
             AND abs(t.amount / coalesce(f.units_per_eur, 1)) < {ARTIFACT_EUR}
             AND f.currency IS NOT NULL
       ) AS n_fx
FROM transactions t
JOIN companies c USING (company_id)
LEFT JOIN prod p ON p.product_id = t.product_id
LEFT JOIN cash.fx f ON f.currency = coalesce(p.currency, c.currency)
WHERE t.date >= DATE '{WINDOW_START}' AND t.date < DATE '{WINDOW_END}'
GROUP BY 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE ar_co AS
SELECT i.company_id,
       coalesce(sum(greatest(i.pending_amount, 0) / coalesce(f.units_per_eur, 1)) FILTER (
           WHERE i.amount > 0
             AND i.pending_amount > 0
             AND i.status NOT IN ('paid', 'cancelled')
             AND i.document_type IN ('invoice', 'invoiceGroup')
             AND i.due_date < DATE '{AS_OF}'
             AND abs(i.pending_amount / coalesce(f.units_per_eur, 1)) < {ARTIFACT_EUR}
       ), 0) AS overdue_ar_eur,
       count(*) FILTER (
           WHERE i.amount > 0
             AND i.document_type IN ('invoice', 'invoiceGroup')
             AND i.due_date < DATE '{AS_OF}'
             AND i.due_date >= DATE '2015-01-01'
             AND i.status <> 'cancelled'
       ) AS n_ar_due,
       count(*) FILTER (
           WHERE i.amount > 0
             AND i.pending_amount > 0
             AND i.status NOT IN ('paid', 'cancelled')
             AND i.document_type IN ('invoice', 'invoiceGroup')
             AND i.due_date < DATE '{AS_OF}'
             AND abs(i.pending_amount / coalesce(f.units_per_eur, 1)) < {ARTIFACT_EUR}
       ) AS n_ar_overdue
FROM invoices i
LEFT JOIN cash.fx f ON f.currency = coalesce(i.currency, 'EUR')
GROUP BY 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE inflow12 AS
SELECT company_id, sum(inflow_eur) AS inflow_12m_eur
FROM cash.cash_metrics
WHERE month_start >= DATE '2025-09-01' AND month_start <= DATE '2026-08-01'
GROUP BY 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE debt_flags AS
SELECT company_id,
       bool_or(type = 'factoring') AS has_factoring,
       bool_or(type = 'lineofcredit') AS has_line,
       bool_or(type = 'confirming') AS has_confirming,
       bool_or(type = 'loan') AS has_loan
FROM debt_products
GROUP BY 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE saving_flags AS
SELECT DISTINCT company_id, TRUE AS has_saving
FROM banking_products WHERE type = 'saving'
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE need AS
SELECT
    c.company_id,
    c.group_id,
    coalesce(c.has_invoices, FALSE) AS has_invoices,
    s.company_id IS NOT NULL AS has_cash,
    coalesce(s.has_drift, TRUE) AS has_drift,
    s.cash_eur,
    s.median_outflow,
    s.coverage,
    s.diagnosis,
    s.runway_months,
    coalesce(fx.fx_annual_eur, 0) AS fx_annual_eur,
    coalesce(fx.n_fx, 0) AS n_fx,
    coalesce(ar.overdue_ar_eur, 0) AS overdue_ar_eur,
    CASE WHEN ar.n_ar_due > 0 THEN ar.n_ar_overdue::DOUBLE / ar.n_ar_due ELSE 0 END AS late_share_ar,
    coalesce(ar.n_ar_due, 0) AS n_ar_due,
    coalesce(inf.inflow_12m_eur, 0) AS inflow_12m_eur,
    coalesce(d.has_factoring, FALSE) AS has_factoring,
    coalesce(d.has_line, FALSE) AS has_line,
    coalesce(d.has_confirming, FALSE) AS has_confirming,
    coalesce(sv.has_saving, FALSE) AS has_saving,

    coalesce(s.company_id IS NOT NULL AND NOT s.has_drift
             AND s.median_outflow > 0 AND s.cash_eur > 2 * s.median_outflow, FALSE) AS elig_yield,
    CASE WHEN s.median_outflow > 0 AND NOT s.has_drift
         THEN greatest(s.cash_eur - 2 * s.median_outflow, 0) ELSE 0 END AS excess_eur,

    coalesce(fx.fx_annual_eur, 0) > 0 AS elig_fx,

    coalesce(s.company_id IS NOT NULL AND NOT s.has_drift AND (
                 s.cash_eur < 0 OR s.coverage < 6
                 OR s.diagnosis IN ('caja negativa', 'colchón crítico', 'deterioro estructural')
             ), FALSE) AS elig_reserve_broad,
    coalesce(s.company_id IS NOT NULL AND NOT s.has_drift
             AND NOT (s.median_outflow > 0 AND s.cash_eur > 2 * s.median_outflow)
             AND (s.cash_eur < 0 OR s.coverage < 2
                  OR s.diagnosis IN ('caja negativa', 'colchón crítico', 'deterioro estructural')),
             FALSE) AS elig_reserve,
    CASE WHEN s.median_outflow > 0
         THEN s.median_outflow * least(greatest(6 - coalesce(s.coverage, 6), 0.5), 6) * 0.5
         ELSE 0 END AS linea_eur,
    CASE WHEN s.cash_eur < 0 THEN -s.cash_eur
         WHEN s.median_outflow > 0 AND s.cash_eur < 2 * s.median_outflow
         THEN greatest(2 * s.median_outflow - s.cash_eur, 0)
         ELSE 0 END AS gap_eur,

    coalesce(c.has_invoices AND coalesce(ar.overdue_ar_eur, 0) > 50000, FALSE) AS elig_factoring,
    coalesce(c.has_invoices AND (
                 (CASE WHEN ar.n_ar_due > 0 THEN ar.n_ar_overdue::DOUBLE / ar.n_ar_due ELSE 0 END) > 0.15
                 OR coalesce(ar.overdue_ar_eur, 0) > 200000
             ), FALSE) AS elig_credit
FROM companies c
LEFT JOIN cash.cash_summary s USING (company_id)
LEFT JOIN fx_co fx USING (company_id)
LEFT JOIN ar_co ar USING (company_id)
LEFT JOIN inflow12 inf USING (company_id)
LEFT JOIN debt_flags d USING (company_id)
LEFT JOIN saving_flags sv USING (company_id)
""")

need = q("SELECT * FROM need")
print("panel", need.shape, "| empresas", need["company_id"].n_unique())

# Reproducir las bases publicadas
check = q("""
SELECT
    count(*) FILTER (WHERE elig_yield) AS surplus_n,
    sum(excess_eur) FILTER (WHERE elig_yield) / 1e6 AS excess_m,
    count(*) FILTER (WHERE elig_fx) AS fx_n,
    sum(fx_annual_eur) FILTER (WHERE elig_fx) / 1e6 AS fx_annual_m,
    count(*) FILTER (WHERE NOT has_drift AND cash_eur < 0) AS deficit_n,
    -sum(cash_eur) FILTER (WHERE NOT has_drift AND cash_eur < 0) / 1e6 AS hole_m
FROM need
""")
display(check)
print("esperado: surplus 370 / 344 M€ · FX 2.108 M€ · déficit 41 / 11,38 M€")


### Conclusión — panel

El panel reproduce las bases de `analysis/monetizacion.py`: **370** empresas con excedente, **344 M€** ociosos; **2.108 M€/año** de flujo cruzado de divisa; **41** agujeros actuales por **11,38 M€**.

Eso es empresa a empresa. La sección siguiente netea a nivel de grupo: el excedente de una filial cubre el agujero de otra antes de barrer o pedir crédito.

Elegibilidad usada a partir de aquí:

- `elig_yield` / `elig_fx`: reglas canónicas medidas.
- `elig_reserve`: **estrecha** y excluye yield (caja negativa, cobertura < 2 meses o diagnóstico de tensión). `elig_reserve_broad` (cobertura < 6) se guarda como sensibilidad: cubre ~992 empresas y no es un producto, es casi el universo.
- `elig_factoring` / `elig_credit`: umbrales de la SPA (50 k€ / 15 % late o 200 k€), sobre AR en EUR, no sobre el panel de research.



## 3b. Movimiento de capital del grupo

179 de 250 grupos tienen más de una sociedad. Si una tiene excedente y otra agujero, el CFO no compra un SKU: traspasa caja. FX no se netea (es flujo de moneda). El factoring de cobros estructurales sigue; el de liquidez de agujero, no cuando la hermana puede pagar.


In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE gnet AS
SELECT
    group_id,
    count(*) AS n_cos,
    count(*) FILTER (WHERE elig_yield) AS n_surplus,
    count(*) FILTER (WHERE NOT has_drift AND cash_eur < 0) AS n_hole,
    coalesce(sum(excess_eur) FILTER (WHERE elig_yield), 0) AS excess_pool,
    coalesce(-sum(cash_eur) FILTER (WHERE NOT has_drift AND cash_eur < 0), 0) AS hole_pool
FROM need
GROUP BY 1
""")

print(q("""
SELECT
  count(*) n_groups,
  count(*) FILTER (WHERE n_cos > 1) multi,
  count(*) FILTER (WHERE n_cos > 1 AND n_surplus > 0 AND n_hole > 0) both_sides,
  count(*) FILTER (WHERE n_hole > 0 AND excess_pool >= hole_pool AND n_cos > 1) full_cover,
  round(sum(least(excess_pool, hole_pool)) FILTER (WHERE n_cos > 1) / 1e6, 2) moved_m,
  round(sum(excess_pool)/1e6, 1) excess_gross_m,
  round(sum(greatest(excess_pool - hole_pool, 0))/1e6, 1) excess_net_m,
  round(sum(hole_pool)/1e6, 2) hole_gross_m,
  round(sum(greatest(hole_pool - excess_pool, 0))/1e6, 2) hole_net_m
FROM gnet
"""))

print("\ngrupos mixtos (excedente + agujero)")
print(q("""
SELECT group_id, n_cos, n_surplus, n_hole,
       round(excess_pool/1e6, 2) excess_m,
       round(hole_pool/1e6, 2) hole_m,
       round(least(excess_pool, hole_pool)/1e6, 2) moved_m
FROM gnet
WHERE n_cos > 1 AND n_surplus > 0 AND n_hole > 0
ORDER BY least(excess_pool, hole_pool) DESC
"""))


### Conclusión — grupo

- **13** grupos tienen excedente y agujero a la vez. Moverían **3,33 M€**. No es yield ni crédito: es pooling.
- Excedente colocable: 344 → **341 M€**. Take yield @ 35 %: 1,20 → **1,19 M€**. El TAM casi no se mueve.
- Agujero: 11,38 → **8,05 M€**. **7** agujeros (1,33 M€) los cubre la hermana; no son demanda de crédito.
- **25** empresas yield tienen hermana en números rojos: no barrer a ciegas.
- FX no se netea. Factoring-como-liquidez: 4 de 26 agujeros factorables cubiertos internamente; 22 siguen con ~129 M€ de AR.

La primera acción del producto no es un SKU de la SPA. Es decirle al grupo que mueva caja.


In [ ]:
SKUS = [
    {"id": "yield", "label": "Yield · excedente", "flag": "elig_yield", "notional": "excess_eur", "bps": YIELD_BPS, "recurring": True},
    {"id": "fx", "label": "FX Shield", "flag": "elig_fx", "notional": "fx_annual_eur", "bps": FX_BPS, "recurring": True},
    {"id": "reserve", "label": "Reserve · línea", "flag": "elig_reserve", "notional": "linea_eur", "bps": RESERVE_BPS, "recurring": True},
    {"id": "factoring", "label": "Factoring", "flag": "elig_factoring", "notional": "overdue_ar_eur", "bps": FACTORING_BPS, "recurring": False},
    {"id": "credit", "label": "Seguro de impago", "flag": "elig_credit", "notional": "inflow_12m_eur", "bps": None, "recurring": True},
]


def sku_stats(cap_notional: bool = False) -> pl.DataFrame:
    rows = []
    for s in SKUS:
        sub = need.filter(pl.col(s["flag"]))
        n = sub.height
        if s["id"] == "credit":
            prem = sub.select(
                (pl.col("inflow_12m_eur").clip(upper_bound=CAP if cap_notional else None)
                 * INSURANCE_PREM * pl.col("late_share_ar")).alias("prem")
            )["prem"]
            notional = prem.sum()  # prima, no inflow
            take = notional * INSURANCE_REFERRAL
            med = sub["late_share_ar"].median()
            p90 = sub["late_share_ar"].quantile(0.9)
            cap_share = None
        else:
            raw = sub[s["notional"]].clip(lower_bound=0)
            capped = raw.clip(upper_bound=CAP) if cap_notional else raw
            notional = capped.sum()
            take = notional * s["bps"]
            med = raw.median()
            p90 = raw.quantile(0.9)
            cap_share = 1 - (capped.sum() / raw.sum()) if cap_notional and raw.sum() else 0.0
        top_k = max(1, n // 10)
        if s["id"] == "credit":
            ordered = prem.sort(descending=True)
            top_share = ordered.head(top_k).sum() / take * INSURANCE_REFERRAL if take else None
            # top_share of take = top of prem * referral / take = top of prem / prem
            top_share = float(ordered.head(top_k).sum() / notional) if notional else None
        else:
            ordered = (capped if cap_notional else raw).sort(descending=True)
            top_share = float(ordered.head(top_k).sum() / notional) if notional else None
        rows.append({
            "sku": s["id"],
            "label": s["label"],
            "n": n,
            "notional_m": (notional or 0) / 1e6,
            "take_100_m": (take or 0) / 1e6,
            "take_25_m": (take or 0) * 0.25 / 1e6,
            "take_35_m": (take or 0) * 0.35 / 1e6,
            "take_45_m": (take or 0) * 0.45 / 1e6,
            "med": med,
            "p90": p90,
            "p90_med": (p90 / med) if med else None,
            "top10_share": top_share,
            "recurring": s["recurring"],
            "cut_by_cap": cap_share,
        })
    return pl.DataFrame(rows)


print("Take independiente, notional crudo (sin tope por empresa)")
display(sku_stats(False))
print("\nTake independiente, notional robusto (tope 100 M€ / empresa = umbral de artefacto)")
display(sku_stats(True))


## 4. Yield · colocación de excedente



In [ ]:
y = need.filter(pl.col("elig_yield"))
print("n", y.height, "| excedente", meur(y["excess_eur"].sum() / 1e6, 1),
      "| mediana", money(y["excess_eur"].median()),
      "| P90", money(y["excess_eur"].quantile(0.9)))
print("sin cuenta saving", y.filter(~pl.col("has_saving")).height,
      "| con saving", y.filter(pl.col("has_saving")).height)
print("take Embat @35 %", meur(y["excess_eur"].sum() * YIELD_BPS * 0.35 / 1e6),
      "| bruto CFO @2,5 % ×35 %", meur(y["excess_eur"].sum() * CFO_GROSS * 0.35 / 1e6),
      "| neto CFO @1,5 % ×35 %", meur(y["excess_eur"].sum() * 0.015 * 0.35 / 1e6))

buckets = q("""
SELECT CASE
         WHEN excess_eur < 5e4 THEN '1. < 50 k€'
         WHEN excess_eur < 1e5 THEN '2. 50–100 k€'
         WHEN excess_eur < 2.5e5 THEN '3. 100–250 k€'
         WHEN excess_eur < 1e6 THEN '4. 250 k€–1 M€'
         WHEN excess_eur < 5e6 THEN '5. 1–5 M€'
         ELSE '6. > 5 M€'
       END AS tramo,
       count(*) AS n,
       sum(excess_eur) / 1e6 AS excess_m
FROM need WHERE elig_yield
GROUP BY 1 ORDER BY 1
""")
display(buckets)

fig = go.Figure(go.Bar(
    x=buckets["tramo"].to_list(), y=buckets["n"].to_list(),
    marker_color=COLORS["blue"],
    text=[f"{n} · {m:.0f} M€" for n, m in zip(buckets["n"], buckets["excess_m"])],
    textposition="outside", cliponaxis=False,
))
fig.update_layout(title="Yield: empresas y notional por tramo de excedente",
                  yaxis_title="Empresas", height=380, margin=dict(t=50, b=40))
fig.show()


### Conclusión — yield

- Reproduce la cifra del pitch: **370 empresas, 344 M€**, mediana **108.720 €**, P90 **2,2 M€** (×20 la mediana). Tras pooling: **360 / 341 M€**, take **1,19 M€/año**. El 10 % de elegibles concentra ~73 % del take: es un módulo de cartera con cola, no un producto de unas pocas cuentas.
- **25** de las 370 tienen una hermana en agujero: primero se mueve capital, no se barre.
- **368 / 370 no tienen cuenta `saving`.** El riel de ejecución no existe en el dataset. X-Ray no «detecta un depósito»; detecta excedente sobre cuentas corrientes y tiene que llevarlo a un partner.
- Prioridad alta: base medida, recurrente, alineada con el pitch, espacio en blanco casi total.



## 5. FX Shield · cobertura de divisa



In [ ]:
fx = need.filter(pl.col("elig_fx"))
print("n con FX > 0", fx.height,
      "| notional anual", meur(fx["fx_annual_eur"].sum() / 1e6, 1),
      "| mediana", money(fx["fx_annual_eur"].median()),
      "| P90", money(fx["fx_annual_eur"].quantile(0.9)))
print("n > 10 k€", fx.filter(pl.col("fx_annual_eur") > 1e4).height,
      "| > 100 k€", fx.filter(pl.col("fx_annual_eur") > 1e5).height,
      "| > 1 M€", fx.filter(pl.col("fx_annual_eur") > 1e6).height)
print("take @35 %", meur(fx["fx_annual_eur"].sum() * FX_BPS * 0.35 / 1e6))

# El proxy de la SPA (n_tx > 50) frente a exposición real
proxy = q("""
WITH ntx AS (
    SELECT company_id, sum(n_tx) AS n_tx
    FROM cash.cash_metrics
    GROUP BY 1
)
SELECT
    count(*) FILTER (WHERE coalesce(n.n_tx, 0) > 50) AS proxy_spa,
    count(*) FILTER (WHERE p.elig_fx) AS fx_real,
    count(*) FILTER (WHERE coalesce(n.n_tx, 0) > 50 AND p.elig_fx) AS ambos,
    count(*) FILTER (WHERE coalesce(n.n_tx, 0) > 50 AND NOT p.elig_fx) AS proxy_sin_fx,
    count(*) FILTER (WHERE coalesce(n.n_tx, 0) <= 50 AND p.elig_fx) AS fx_sin_proxy
FROM need p
LEFT JOIN ntx n USING (company_id)
""")
print("\nproxy SPA n_tx>50 vs FX real (moneda producto ≠ moneda empresa)")
display(proxy)

fb = q("""
SELECT CASE
         WHEN fx_annual_eur < 1e4 THEN '1. < 10 k€'
         WHEN fx_annual_eur < 1e5 THEN '2. 10–100 k€'
         WHEN fx_annual_eur < 1e6 THEN '3. 100 k€–1 M€'
         WHEN fx_annual_eur < 1e7 THEN '4. 1–10 M€'
         ELSE '5. > 10 M€'
       END AS tramo,
       count(*) AS n,
       sum(fx_annual_eur) / 1e6 AS fx_m
FROM need WHERE elig_fx
GROUP BY 1 ORDER BY 1
""")
display(fb)
fig = go.Figure(go.Bar(
    x=fb["tramo"].to_list(), y=fb["n"].to_list(),
    marker_color=COLORS["cyan"],
    text=[f"{n} · {m:.0f} M€" for n, m in zip(fb["n"], fb["fx_m"])],
    textposition="outside", cliponaxis=False,
))
fig.update_layout(title="FX: empresas y notional anual por tramo",
                  yaxis_title="Empresas", height=380, margin=dict(t=50, b=40))
fig.show()


### Conclusión — FX

- **261** empresas con flujo cruzado real, **2.108 M€/año**. Mediana **796 k€**, P90 **19 M€**. Take central **1,11 M€/año** (15 pb × 35 %). Reproduce la monetización.
- 197 tienen más de 100 k€/año: esa es la cola accionable. 33 empresas tienen FX de adorno (< 10 k€).
- El proxy de la SPA (`n_tx > 50`) **no es exposición a divisa**: marca actividad, no moneda. No usarlo para recomendar FX Shield.
- No se canibaliza con yield (solo 56 empresas en ambos). Segundo producto del pitch, misma clase de evidencia.



## 6. Reserve · financiación preventiva



In [ ]:
rb = need.filter(pl.col("elig_reserve_broad"))
rt = need.filter(pl.col("elig_reserve"))
print("broad (cobertura < 6 o tensión)", rb.height,
      "linea cruda", meur(rb["linea_eur"].sum() / 1e6, 0),
      "linea tope 100 M€", meur(rb["linea_eur"].clip(upper_bound=CAP).sum() / 1e6, 1))
print("tight (sin yield, cobertura < 2 o tensión)", rt.height,
      "linea cruda", meur(rt["linea_eur"].sum() / 1e6, 0),
      "linea tope 100 M€", meur(rt["linea_eur"].clip(upper_bound=CAP).sum() / 1e6, 1),
      "take crudo @35 %", meur(rt["linea_eur"].sum() * RESERVE_BPS * 0.35 / 1e6, 1),
      "take tope @35 %", meur(rt["linea_eur"].clip(upper_bound=CAP).sum() * RESERVE_BPS * 0.35 / 1e6))

hole = need.filter((~pl.col("has_drift")) & (pl.col("cash_eur") < 0))
print("\nsolo agujero actual: n", hole.height,
      "notional", meur((-hole["cash_eur"]).sum() / 1e6),
      "take 50 pb @35 %", meur((-hole["cash_eur"]).sum() * RESERVE_BPS * 0.35 / 1e6, 3))

print("tight ∩ póliza ya viva", rt.filter(pl.col("has_line")).height,
      "| origination (sin póliza)", rt.filter(~pl.col("has_line")).height)

print("\ncolas que rompen la fórmula (median_outflow de cientos de M€/mes)")
display(q("""
SELECT company_id, round(median_outflow/1e6, 1) AS outflow_m,
       round(coverage, 3) AS coverage, diagnosis,
       round(linea_eur/1e6, 1) AS linea_m
FROM need
WHERE elig_reserve
ORDER BY linea_eur DESC
LIMIT 8
"""))

print("\ndiagnóstico dentro de tight")
display(q("""
SELECT diagnosis, count(*) AS n,
       sum(linea_eur)/1e6 AS linea_m,
       sum(least(linea_eur, 1e8))/1e6 AS linea_cap_m
FROM need WHERE elig_reserve
GROUP BY 1 ORDER BY n DESC
"""))


### Conclusión — reserve

- La regla ancha (cobertura < 6 meses) coge **992** empresas: casi todo el universo fiable. No es un SKU, es «quien no es rico».
- La regla estrecha (sin yield + colchón < 2 meses o diagnóstico de tensión) coge **805**. El n es real: hay mucha tesorería fina. El **notional de línea no lo es**: `median_outflow × clip(6 − coverage) × 0,5` produce **21.000 M€** porque unas pocas series tienen gasto mensual de cientos de millones (COMP_1185: 2,9 B€/mes). Eso no es una pyme colocable; es cola del generador que pasó el filtro de 100 M€ **por movimiento**.
- Con tope 100 M€/empresa el take central sigue en ~**8 M€**, dominado por la cola. Sobre los **41 agujeros brutos (11,38 M€)** el take es **0,02 M€**. Tras pooling quedan **8,05 M€** en 34 agujeros (7 los cubre la hermana). Esa es la cifra defendible de demanda de crédito, y es la razón por la que el pitch no lidera con marketplace.
- 158 de las 805 ya tienen póliza. Origination pura: 647, pero sin un notional creíble.
- Prioridad de **pitch: baja**. Prioridad de **producto de aviso** (negociar línea antes del agujero): media, como acción, no como P&L.



## 7. Factoring · anticipo de cobros



In [ ]:
fa = need.filter(pl.col("elig_factoring"))
print("n", fa.height, "de", need.filter(pl.col("has_invoices")).height, "con ERP")
print("AR vencido", meur(fa["overdue_ar_eur"].sum() / 1e6, 1),
      "| mediana", money(fa["overdue_ar_eur"].median()),
      "| P90", money(fa["overdue_ar_eur"].quantile(0.9)))
print("take 1,5 % @100 %", meur(fa["overdue_ar_eur"].sum() * FACTORING_BPS / 1e6),
      "| @35 %", meur(fa["overdue_ar_eur"].sum() * FACTORING_BPS * 0.35 / 1e6),
      "| one-shot, no renta")
print("ya tienen factoring", fa.filter(pl.col("has_factoring")).height,
      "| espacio en blanco", fa.filter(~pl.col("has_factoring")).height)
print("sin ERP, no se puede ofrecer", need.filter(~pl.col("has_invoices")).height)

ab = q("""
SELECT CASE
         WHEN overdue_ar_eur < 5e4 THEN '1. < 50 k€ (fuera)'
         WHEN overdue_ar_eur < 2e5 THEN '2. 50–200 k€'
         WHEN overdue_ar_eur < 1e6 THEN '3. 200 k€–1 M€'
         WHEN overdue_ar_eur < 5e6 THEN '4. 1–5 M€'
         ELSE '5. > 5 M€'
       END AS tramo,
       count(*) AS n,
       sum(overdue_ar_eur) / 1e6 AS ar_m
FROM need WHERE has_invoices
GROUP BY 1 ORDER BY 1
""")
display(ab)
fig = go.Figure(go.Bar(
    x=ab["tramo"].to_list(), y=ab["n"].to_list(),
    marker_color=COLORS["amber"],
    text=[f"{n} · {m:.0f} M€" for n, m in zip(ab["n"], ab["ar_m"])],
    textposition="outside", cliponaxis=False,
))
fig.update_layout(title="AR vencido (EUR) en empresas con ERP, umbral 50 k€ = factoring",
                  yaxis_title="Empresas", height=380, margin=dict(t=50, b=40))
fig.show()


### Conclusión — factoring

- **461** empresas con AR vencido > 50 k€ (de 785 con ERP; 637 con algún vencido). Notional **1.283 M€**. Mediana **570 k€**, P90 **5,9 M€**. El 10 % de elegibles carga ~69 % del take.
- Take **6,74 M€** al 35 % si se factoriza todo el vencido a 1,5 %. Es **one-shot**, no comparable 1:1 con los 1,20 M€ recurrentes de yield. Y asume que el vencido es cedible (calidad de deudor, recurso, que no esté ya descontado).
- Espacio en blanco: **454 / 461** no tienen `debt_products.type = factoring` (solo 19 empresas en toda la cartera lo tienen). El producto casi no está implantado.
- Bloqueo duro: **501 empresas sin ERP**. Ahí X-Ray no puede recomendar factoring; puede recomendar conectar el ERP.
- Prioridad: **tercera**, como producto de acción sobre cobros, no como línea de P&L del pitch. La base de AR sí está medida; el take no.



## 8. Credit · seguro de impago



In [ ]:
cr = need.filter(pl.col("elig_credit"))
prem = cr["inflow_12m_eur"] * INSURANCE_PREM * cr["late_share_ar"]
prem_cap = cr["inflow_12m_eur"].clip(upper_bound=CAP) * INSURANCE_PREM * cr["late_share_ar"]
print("n", cr.height, "| late_share mediana", f"{100*cr['late_share_ar'].median():.1f} %",
      "| P90", f"{100*cr['late_share_ar'].quantile(0.9):.0f} %")
print("prima cruda", meur(prem.sum() / 1e6, 1),
      "| referral 20 % @35 %", meur(prem.sum() * INSURANCE_REFERRAL * 0.35 / 1e6, 1))
print("prima con inflow tope 100 M€", meur(prem_cap.sum() / 1e6),
      "| referral @35 %", meur(prem_cap.sum() * INSURANCE_REFERRAL * 0.35 / 1e6))
print("solape con factoring", need.filter(pl.col("elig_credit") & pl.col("elig_factoring")).height,
      "| solo seguro", need.filter(pl.col("elig_credit") & ~pl.col("elig_factoring")).height,
      "| solo factoring", need.filter(~pl.col("elig_credit") & pl.col("elig_factoring")).height)

print("\nquién entra por late_share > 15 % vs por AR > 200 k€")
display(q("""
SELECT
    count(*) FILTER (WHERE late_share_ar > 0.15) AS por_late,
    count(*) FILTER (WHERE overdue_ar_eur > 200000) AS por_ar,
    count(*) FILTER (WHERE late_share_ar > 0.15 AND overdue_ar_eur > 200000) AS ambos,
    count(*) FILTER (WHERE late_share_ar > 0.15 AND overdue_ar_eur <= 200000) AS late_sin_ar_grande
FROM need WHERE elig_credit
"""))


### Conclusión — seguro de impago

- **484** empresas (ERP + late_share > 15 % o AR > 200 k€). Mediana de morosidad **36 %** entre elegibles: el umbral 15 % es laxo en este dataset.
- **405 / 484** también son elegibles a factoring. Son la misma cola de cobros, dos envoltorios (anticipo vs prima).
- La prima `inflow × 0,5 % × late_share` **revienta con las mismas series de gasto enorme** (prima cruda 216 M€ → referral central 15 M€). Con tope 100 M€ de inflow, el take central cae a **~0,76 M€**: por debajo de yield y FX, y con tres hipótesis apiladas (prima, referral, adopción).
- Prioridad: **cuarta**. Mismo comprador de ejecución (aseguradora), misma dependencia de ERP, menor take robusto y más solape que factoring. Sirve como upsell sobre la cola de cobros, no como SKU de apertura.



## 9. Espacio en blanco vs producto ya contratado



In [ ]:
white = q("""
SELECT
    count(*) FILTER (WHERE elig_yield) AS yield_n,
    count(*) FILTER (WHERE elig_yield AND NOT has_saving) AS yield_sin_saving,
    count(*) FILTER (WHERE elig_factoring) AS fac_n,
    count(*) FILTER (WHERE elig_factoring AND NOT has_factoring) AS fac_sin_producto,
    count(*) FILTER (WHERE elig_reserve) AS res_n,
    count(*) FILTER (WHERE elig_reserve AND has_line) AS res_con_poliza,
    count(*) FILTER (WHERE elig_reserve AND NOT has_line) AS res_sin_poliza,
    count(*) FILTER (WHERE elig_reserve AND NOT has_drift AND cash_eur < 0 AND NOT has_line) AS agujero_sin_poliza,
    count(*) FILTER (WHERE elig_credit) AS cred_n
FROM need
""")
display(white)

print("agujeros actuales sin póliza (la demanda de crédito medible)")
display(q("""
SELECT company_id, round(cash_eur, 0) AS cash_eur, diagnosis, has_line, has_factoring
FROM need
WHERE NOT has_drift AND cash_eur < 0
ORDER BY cash_eur
"""))


### Conclusión — espacio en blanco

- Yield es el hueco más limpio: **368/370** sin `saving`. Embat no tiene que desplazar un depósito; tiene que crear el riel.
- Factoring: **454/461** sin producto. Confirming ya está en 70 empresas; factoring, en 19. El dataset está sesgado a préstamo/póliza, no a cesión de cobros.
- Reserve: 158/805 ya tienen póliza (renovación / subir límite), 647 serían origination. De los **41 agujeros**, la pregunta útil es cuántos no tienen línea —eso es demanda, no un marketplace de 800 nombres.



## 10. Solape



In [ ]:
flags = ["elig_yield", "elig_fx", "elig_reserve", "elig_factoring", "elig_credit"]
labels = ["yield", "fx", "reserve", "factoring", "credit"]

n_sku = need.select(
    pl.sum_horizontal([pl.col(f).cast(pl.Int64) for f in flags]).alias("n_sku")
)
print("¿a cuántos SKUs es elegible cada empresa?")
display(n_sku["n_sku"].value_counts().sort("n_sku"))

rows = []
for f, a in zip(flags, labels):
    row = {"sku": a}
    for g, b in zip(flags, labels):
        row[b] = need.filter(pl.col(f) & pl.col(g)).height
    rows.append(row)
pair = pl.DataFrame(rows)
print("\nmatriz de co-elegibilidad (tight reserve, yield ∩ reserve = 0 por construcción)")
display(pair)

print("\nempresas en 3+ SKUs: las que hinchan un TAM independiente")
multi = need.filter(pl.sum_horizontal([pl.col(f).cast(pl.Int64) for f in flags]) >= 3)
print("n", multi.height,
      "| con ERP", multi.filter(pl.col("has_invoices")).height,
      "| con FX", multi.filter(pl.col("elig_fx")).height)


### Conclusión — solape

- 57 empresas a 0 SKUs (sin caja fiable ni ERP accionable). 564 a uno. **327 a tres, 80 a cuatro**: el TAM independiente suma la misma tesorería varias veces.
- Yield ∩ reserve = 0 por construcción (el excedente no origina línea).
- Yield ∩ FX = **56**: se suman, no se restan.
- Factoring ∩ seguro = **405**: casi el mismo set. El waterfall no puede contar 6,7 M€ + 0,76 M€ como si fueran carteras distintas.
- Reserve ∩ FX = 175, reserve ∩ factoring = 320: la empresa tensa suele tener también cobros vencidos o divisa. Eso es una **cola de decisiones**, no cinco productos independientes.



## 11. Prioridad incremental

Greedy por take central **robusto** (notional tope 100 M€/empresa), con reglas de canibalización:

1. Yield y FX se suman (bases distintas).
2. Reserve no se cuenta sobre elegibles a yield.
3. Seguro se cuenta **solo en empresas no elegibles a factoring** (upsell residual).
4. Factoring es one-shot: se reporta aparte de la renta anual.



In [ ]:
def take_of(df: pl.DataFrame, sku_id: str, cap: bool = True) -> float:
    # Take central @ 35 %. Yield/FX/factoring ya vienen limpios por movimiento.
    # Reserve y seguro sí se topan: su notional es fórmula sobre colas enormes.
    if df.height == 0:
        return 0.0
    cap_here = cap and sku_id in {"reserve", "credit"}
    if sku_id == "yield":
        return df["excess_eur"].sum() * YIELD_BPS * 0.35
    if sku_id == "fx":
        return df["fx_annual_eur"].sum() * FX_BPS * 0.35
    if sku_id == "reserve":
        n = df["linea_eur"].clip(upper_bound=CAP) if cap_here else df["linea_eur"]
        return n.sum() * RESERVE_BPS * 0.35
    if sku_id == "factoring":
        return df["overdue_ar_eur"].sum() * FACTORING_BPS * 0.35
    if sku_id == "credit":
        inf = df["inflow_12m_eur"].clip(upper_bound=CAP) if cap_here else df["inflow_12m_eur"]
        return (inf * INSURANCE_PREM * df["late_share_ar"]).sum() * INSURANCE_REFERRAL * 0.35
    raise KeyError(sku_id)


waterfall = []

# step 1 yield
step = need.filter(pl.col("elig_yield"))
waterfall.append({"paso": 1, "sku": "yield", "n_nuevo": step.height,
                  "take_m": take_of(step, "yield") / 1e6, "clase": "renta medida",
                  "incremental": True})

# step 2 fx (all fx, including those with yield)
step = need.filter(pl.col("elig_fx"))
waterfall.append({"paso": 2, "sku": "fx", "n_nuevo": step.height,
                  "take_m": take_of(step, "fx") / 1e6, "clase": "renta medida",
                  "incremental": True})

# step 3 factoring
step = need.filter(pl.col("elig_factoring"))
waterfall.append({"paso": 3, "sku": "factoring", "n_nuevo": step.height,
                  "take_m": take_of(step, "factoring") / 1e6, "clase": "one-shot, AR medido",
                  "incremental": True})

# step 4 credit residual (not in factoring)
step = need.filter(pl.col("elig_credit") & ~pl.col("elig_factoring"))
waterfall.append({"paso": 4, "sku": "credit (residual)", "n_nuevo": step.height,
                  "take_m": take_of(step, "credit") / 1e6, "clase": "renta hipótesis, residual",
                  "incremental": True})
step_full = need.filter(pl.col("elig_credit"))
waterfall.append({"paso": 4, "sku": "credit (bruto, solapa)", "n_nuevo": step_full.height,
                  "take_m": take_of(step_full, "credit") / 1e6, "clase": "no sumar al waterfall",
                  "incremental": False})

# step 5 reserve tight, robust cap; and hole-only as alternative
step = need.filter(pl.col("elig_reserve"))
waterfall.append({"paso": 5, "sku": "reserve (línea, tope 100 M€)", "n_nuevo": step.height,
                  "take_m": take_of(step, "reserve") / 1e6, "clase": "no defendible como P&L",
                  "incremental": False})
holes = need.filter((~pl.col("has_drift")) & (pl.col("cash_eur") < 0))
waterfall.append({"paso": 5, "sku": "reserve (solo agujero 41)", "n_nuevo": holes.height,
                  "take_m": ((-holes["cash_eur"]).sum() * RESERVE_BPS * 0.35) / 1e6,
                  "clase": "demanda de crédito medida",
                  "incremental": True})

wf = pl.DataFrame(waterfall)
display(wf)

renta = wf.filter(pl.col("sku").is_in(["yield", "fx"]))["take_m"].sum()
print(f"renta medida yield+FX @35 %: {renta:.2f} M€/año")
oneshot = wf.filter(pl.col("sku") == "factoring")["take_m"][0]
print(f"factoring one-shot @35 %: {oneshot:.2f} M€ (no anualizar)")
print(f"seguro residual @35 %: {wf.filter(pl.col('sku') == 'credit (residual)')['take_m'][0]:.3f} M€")
print(f"agujero crédito @35 %: {wf.filter(pl.col('sku') == 'reserve (solo agujero 41)')['take_m'][0]:.3f} M€")

inc = wf.filter(pl.col("incremental"))
fig = go.Figure(go.Waterfall(
    x=inc["sku"].to_list(),
    y=inc["take_m"].to_list(),
    measure=["relative"] * inc.height,
    connector={"line": {"color": COLORS["slate"]}},
    increasing={"marker": {"color": COLORS["blue"]}},
))
fig.update_layout(title="Take central incremental (M€) · solo palos defendibles o residuales",
                  yaxis_title="M€ @ 35 % adopción", height=400, margin=dict(t=50))
fig.show()


### Conclusión — waterfall

Orden por evidencia, no por TAM independiente:

0. **Pooling** en 13 grupos (3,33 M€). No es un SKU.
1. **Yield** +1,19 M€/año (360 empresas netas; bruto 370 / 344 M€). Base medida.
2. **FX** +1,11 M€/año (261, 56 ya en yield). Base medida; no se netea. Renta conjunta **2,30 M€/año**.
3. **Factoring** +6,74 M€ one-shot (461, 454 sin producto). AR medido; take y cesión, hipótesis. No se suma a la renta.
4. **Seguro residual** (elegible a seguro y no a factoring) es pequeño; el bruto 0,76 M€ solapa con el paso 3.
5. **Reserve** como P&L de línea no entra. La demanda neta son 8,05 M€ (34 agujeros que el grupo no tapa). Como acción de producto («negocia la póliza ya»), sí: 805 empresas tensas, 331 con colchón crítico —salvo si hay caja hermana.

Si se ordenara por take independiente sin tope, reserve (37 M€) y seguro (15 M€) ganarían. Esa ordenación es un error de cola + fórmula, no una oportunidad comercial.



## 12. Sensibilidad de umbrales



In [ ]:
print("factoring: n y notional según umbral de AR vencido")
rows = []
for thr in (10_000, 50_000, 100_000, 200_000, 1_000_000):
    sub = need.filter(pl.col("has_invoices") & (pl.col("overdue_ar_eur") > thr))
    rows.append({"umbral": thr, "n": sub.height,
                 "ar_m": sub["overdue_ar_eur"].sum() / 1e6,
                 "take_35_m": sub["overdue_ar_eur"].sum() * FACTORING_BPS * 0.35 / 1e6})
display(pl.DataFrame(rows))

print("\nFX: n y take según notional mínimo anual")
rows = []
for thr in (0, 10_000, 100_000, 1_000_000):
    sub = need.filter(pl.col("fx_annual_eur") > thr)
    rows.append({"mínimo": thr, "n": sub.height,
                 "fx_m": sub["fx_annual_eur"].sum() / 1e6,
                 "take_35_m": sub["fx_annual_eur"].sum() * FX_BPS * 0.35 / 1e6})
display(pl.DataFrame(rows))

print("\nyield: n y take según meses de colchón (cash > k × outflow)")
rows = []
for k in (1, 2, 3, 6):
    sub = need.filter((~pl.col("has_drift")) & (pl.col("median_outflow") > 0)
                      & (pl.col("cash_eur") > k * pl.col("median_outflow")))
    excess = (sub["cash_eur"] - k * sub["median_outflow"]).clip(lower_bound=0)
    rows.append({"k_meses": k, "n": sub.height, "excess_m": excess.sum() / 1e6,
                 "take_35_m": excess.sum() * YIELD_BPS * 0.35 / 1e6})
display(pl.DataFrame(rows))

print("\nadopción sobre las dos rentas medidas (yield+FX)")
base = need["excess_eur"].sum() * YIELD_BPS + need.filter(pl.col("elig_fx"))["fx_annual_eur"].sum() * FX_BPS
for name, a in ADOPTION.items():
    print(f"  {name:12} {a:.0%}  {base * a / 1e6:.2f} M€/año")


### Conclusión — sensibilidad

- Yield es estable: pasar el colchón de 2 a 3 meses apenas recorta el take (la cola gruesa sigue dentro). Bajar a 1 mes infla n y mezcla tesorería operativa con ocioso.
- FX: recortar a > 100 k€/año deja 197 empresas y casi todo el notional (la cola es el producto). Recortar a 1 M€ pierde n y aún conserva la mayor parte del take.
- Factoring: el umbral 50 k€ de la SPA es arbitrario; a 200 k€ quedan 330 empresas y la mayor parte del AR. El take se mueve poco porque está en la cola > 1 M€.
- Adopción 25/35/45 % sobre yield+FX: **1,65 / 2,31 / 2,97 M€/año**. Esa es la horquilla que se puede poner junto al pitch, no 8–37 M€ de líneas.



## 13. Veredicto



In [ ]:
verdict = pl.DataFrame([
    {
        "prioridad": 0,
        "sku": "pooling",
        "n": "13 grupos mixtos",
        "notional": "3,33 M€ a mover internamente",
        "take_35": "0 (no es un SKU)",
        "incremental": "redirige 7 agujeros y 25 yield",
        "evidencia": "medida (caja de grupo, sin deriva)",
        "riesgo_comercial": "ninguno: es tesorería, no producto",
        "nota": "Primera acción. No barrer ni prestar si hay caja hermana.",
    },
    {
        "prioridad": 1,
        "sku": "yield",
        "n": "360 netas / 370 brutas",
        "notional": "341 M€ netos (344 brutos)",
        "take_35": "1,19 M€/año neto",
        "incremental": "1,19 M€ renta tras pooling",
        "evidencia": "medida (reproduce monetización)",
        "riesgo_comercial": "bajo: riel saving casi inexistente (2/370)",
        "nota": "Abrir con esto, después de mirar el grupo. 25 tienen hermana en agujero.",
    },
    {
        "prioridad": 2,
        "sku": "fx",
        "n": 261,
        "notional": "2.108 M€/año",
        "take_35": "1,11 M€/año",
        "incremental": "1,11 M€ renta (56 ya en yield)",
        "evidencia": "medida (no usar n_tx>50)",
        "riesgo_comercial": "bajo-medio: ejecución vía partner de pagos",
        "nota": "Segundo del pitch. Umbral útil: >100 k€/año (197 empresas).",
    },
    {
        "prioridad": 3,
        "sku": "factoring",
        "n": 461,
        "notional": "1.283 M€ AR vencido",
        "take_35": "6,74 M€ one-shot",
        "incremental": "6,74 M€ no anual; 454 sin producto",
        "evidencia": "AR medido; 1,5 % y cesión, hipótesis",
        "riesgo_comercial": "medio: ERP obligatorio; 501 sin facturas",
        "nota": "Acción sobre cobros, no línea de P&L del pitch. Confirming ya existe (70).",
    },
    {
        "prioridad": 4,
        "sku": "credit",
        "n": 484,
        "notional": "prima robusta ~2 M€",
        "take_35": "0,76 M€/año robusto (15 M€ crudo, no usar)",
        "incremental": "casi todo solapa con factoring (405)",
        "evidencia": "elegibilidad medida; prima apilada",
        "riesgo_comercial": "alto: aseguradora, hipótesis triples",
        "nota": "Upsell de la cola de cobros, no SKU de apertura.",
    },
    {
        "prioridad": 5,
        "sku": "reserve",
        "n": "805 tensas / 34 agujeros netos (41 brutos)",
        "notional": "8,05 M€ agujero neto; 11,38 M€ bruto; 21.000 M€ línea cruda (descartar)",
        "take_35": "0,01 M€ sobre agujero neto; 8–37 M€ fórmula (no pitch)",
        "incremental": "0,01 M€ (7 agujeros los tapa el grupo)",
        "evidencia": "n de tensión medido; notional de línea no",
        "riesgo_comercial": "alto si se vende como marketplace de crédito",
        "nota": "Acción: si el grupo no cubre, negociar póliza. No liderar el P&L.",
    },
])
display(verdict.select(["prioridad", "sku", "n", "notional", "take_35", "incremental", "evidencia"]))
print()
for row in verdict.iter_rows(named=True):
    print(f"{row['prioridad']}. {row['sku']}: {row['nota']}")


### Veredicto

**Añadir, en este orden:** movimiento de capital del grupo → yield → FX → factoring como acción de cobros → seguro como upsell → reserve como aviso, no como marketplace.

La renta defendible junto al módulo SaaS sigue siendo la de `monetizacion.md`, ya neta de pooling: **excedente 1,19 + divisa 1,11 M€/año**. Factoring hincha un one-shot si se toma el 1,5 % del AR vencido; no sustituye esa renta. Reserve y seguro, sin tope por empresa, son artefactos de cola. **7** de **41** agujeros no son crédito: los tapa una hermana.

Regla para la demo de Productos: no mostrar un ranking por comisión simulada de la SPA. Mostrar necesidad tesorera medida, y no recomendar barrido ni crédito si hay caja hermana.

